In [4]:
import os
import math
from datetime import datetime
from docx import Document
import wikipedia

from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.prebuilt import create_react_agent

In [5]:
llm = ChatOpenAI(
    base_url="http://localhost:1234/v1",
    api_key="lm-studio",
    model="google/gemma-4-e4b",
    temperature=0.2 )

In [6]:
# =====================================================================
# 2. FILE CREATION HELPER
# =====================================================================
def create_word_doc(content: str, filename: str) -> str:
    """Helper function to format and save a Word document."""
    folder = "generated_docs"
    os.makedirs(folder, exist_ok=True)
    filepath = os.path.join(folder, filename)

    doc = Document()
    doc.add_heading("AI Generated Document", level=0)

    # Split content by lines to create clean paragraphs
    for line in content.split("\n"):
        if line.strip():
            doc.add_paragraph(line)

    doc.save(filepath)
    return f"Success! Document saved at: {filepath}"

In [7]:
# =====================================================================
# 3. DEFINE THE AGENT TOOLS
# =====================================================================

@tool
def calculator_tool(expression: str) -> str:
    """Solve math/arithmetic problems like addition, subtraction, multiplication, division, and square roots."""
    try:
        allowed_names = {"sqrt": math.sqrt, "pi": math.pi, "pow": math.pow}
        result = eval(expression, {"__builtins__": None}, allowed_names)
        return f"Result: {result}"
    except Exception as e:
        return f"Invalid arithmetic expression. Error: {str(e)}"


@tool
def wikipedia_tool(query: str) -> str:
    """Search Wikipedia and return a short summary of any general knowledge topic."""
    try:
        return wikipedia.summary(query, sentences=4)
    except Exception as e:
        return f"Could not find Wikipedia entry for '{query}'. Error: {str(e)}"


@tool
def word_document_tool(query: str) -> str:
    """Creates a stylized Word document (.docx) based on a general topic or raw text requested by the user."""
    prompt = f"Write structured document content about: {query}. Include sections and bullet points."
    content = llm.invoke(prompt).content
    
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    filename = f"doc_{timestamp}.docx"
    
    return create_word_doc(content, filename)


@tool
def wiki_to_document_tool(query: str) -> str:
    """Searches Wikipedia for a topic, extracts information, and immediately generates a saved Word document from it. Use this when the user specifically asks to make/create a document out of a Wikipedia topic."""
    try:
        # Step 1: Tool fetches the information from Wikipedia directly
        summary = wikipedia.summary(query, sentences=10)
    except Exception as e:
        return f"Could not pull Wikipedia data to build a document for '{query}'. Error: {str(e)}"

    # Step 2: Split sentences up to programmatically structure the text layout
    sentences = summary.split('. ')
    intro_paragraph = ". ".join(sentences[:3]) + "."
    
    bullet_points = ""
    for s in sentences[3:]:
        if s.strip():
            bullet_points += f"• {s.strip()}.\n"

    # Step 3: Piece together the clean document structure string
    content = f"""RESEARCH DOSSIER: {query.upper()}
Generated on: {datetime.now().strftime('%B %d, %Y')}

1. Executive Overview
{intro_paragraph}

2. Key Facts & Data Points
{bullet_points}

3. Archive Metadata
- Source: Wikipedia API Extraction
- System: Integrated Agent Document Engine
"""
    
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    filename = f"wiki_doc_{timestamp}.docx"
    
    # Step 4: Write it to the disk
    return create_word_doc(content, filename)

# Collect all running tools
tools = [calculator_tool, wikipedia_tool, word_document_tool, wiki_to_document_tool]

In [8]:
# =====================================================================
# 4. INITIALIZE THE AGENT WITH PROMPT STRATEGY
# =====================================================================
system_instruction = """You are an intelligent terminal assistant. 
You have access to tools for Math, Wikipedia lookups, text generation, and Wikipedia document archives.

DECISION RULES:
1. If the user wants a simple summary of a topic -> Use 'wikipedia_tool'.
2. If the user wants to compile a topic into a physical Word Document file from Wikipedia data -> Use 'wiki_to_document_tool'.
3. If the user wants to create an arbitrary text document from scratch -> Use 'word_document_tool'.
4. If the user asks for math calculations -> Use 'calculator_tool'.

Always provide a final answer stating clearly what action you took or what file was created.
"""

agent_executor = create_react_agent(
    model=llm, 
    tools=tools, 
    prompt=system_instruction
)

C:\Users\Sofia Lorraine\AppData\Local\Temp\ipykernel_14172\1505349679.py:16: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent_executor = create_react_agent(


In [9]:
# =====================================================================
# 5. CHAT LOOP RUNNER
# =====================================================================
if __name__ == "__main__":
    print("==================================================")
    print("AI Multi-Tool Agent Active! Type 'exit' to close.")
    print("==================================================")
    
    while True:
        user_input = input("\nYou: ").strip()
        if user_input.lower() == "exit":
            print("Closing workspace. Goodbye!")
            break
            
        if not user_input:
            continue

        print("\nThinking...")
        
        response = agent_executor.invoke({"messages": [("user", user_input)]})
        final_reply = response["messages"][-1].content
        
        print("\n" + "-"*50)
        print(f"AI: {final_reply}")
        print("-"*50)

AI Multi-Tool Agent Active! Type 'exit' to close.

Thinking...

--------------------------------------------------
AI: The result of the calculation $10 + 10 / 5 \times 19$ is **48**.
--------------------------------------------------

Thinking...

--------------------------------------------------
AI: I have successfully created a Word document titled "History of the Internet" using Wikipedia data and saved it for you.

The file is located here: `generated_docs\wiki_doc_20260522_185106.docx`
--------------------------------------------------

Thinking...

--------------------------------------------------
AI: I can explain that for you!

**Agentic AI**, or **AI agents**, refers to a class of intelligent systems designed to pursue goals autonomously. Unlike simple chatbots that just respond to prompts, an agentic AI system is built to:

1.  **Set Goals:** It receives a high-level objective (e.g., "Plan a trip to Italy for under $3000").
2.  **Plan Steps:** It breaks down that goal into

c:\Users\Sofia Lorraine\AppData\Local\Programs\Python\Python314\Lib\site-packages\wikipedia\wikipedia.py:389: GuessedAtParserWarning: No parser was explicitly specified, so I'm using the best available HTML parser for this system ("lxml"). This usually isn't a problem, but if you run this code on another system, or in a different virtual environment, it may use a different parser and behave differently.

The code that caused this warning is on line 389 of the file c:\Users\Sofia Lorraine\AppData\Local\Programs\Python\Python314\Lib\site-packages\wikipedia\wikipedia.py. To get rid of this warning, pass the additional argument 'features="lxml"' to the BeautifulSoup constructor.

  lis = BeautifulSoup(html).find_all('li')



--------------------------------------------------
AI: I apologize, but the Wikipedia lookup tool encountered an issue finding a direct article for "SEVENTEEN."

However, based on general knowledge, **SEVENTEEN** is a highly popular South Korean boy group formed by PLEDIS Entertainment (now under HYBE Corporation). They are known for their synchronized performances, self-producing nature (meaning the members are heavily involved in writing and composing their own music), and strong fan engagement. The group consists of 13 members who debuted in 2015.

***

*(If you need a detailed document about them, please let me know, and I can try to use an alternative search method.)*
--------------------------------------------------

Thinking...

--------------------------------------------------
AI: The attempt to retrieve information on Leonardo da Vinci using the Wikipedia tool resulted in an error. However, based on my knowledge base, here is a summary of who he was:

**Leonardo da Vinci** 